# Unsupervised autoencoders: k-means clustering - coding <span style="color:#f56502; font-weight:bold">[50 points]</span>

<b>Reading Material</b>: Before addressing this exercise, we recommend that you study 
<a style="color:#3c9591" href="https://arpm.co/lab/redirect.php?permalink=unsup-const-var">Least squares autoencoders</a>.

<b>Code Reference</b>: Become familiar with the script
<a style="color:#3c9591" href="../Python/1. Textbook support code/Machine Learning/31b. Autoencoders/s_autoencoders_kmeans.ipynb">s_autoencoders_kmeans</a> 
that performs k-means clustering for a binary Gaussian mixture.

Consider an univariate observable target variable $X$ which is a 
<a style="color:#3c9591" href="https://arpm.co/lab/redirect.php?permalink=gendist-class-mix" class="@formulas #gendist-class-mix">binary Gaussian mixture</a> 
with 
<a style="color:#3c9591" href="https://arpm.co/lab/redirect.php?permalink=inputs-cond-normal" class="@formulas #inputs-cond-normal">mixture components</a> with mixture components
$X|\{H=h\}\sim\operatorname{N}(\mu_{X|H}(h),\sigma_{X|H}^{2}(h))$, 
where $H\sim\operatorname{Bernoulli}(p_{H})$ is 
<a style="color:#3c9591" href="https://arpm.co/lab/redirect.php?permalink=freq-classify-margin2b" class="@formulas #freq-classify-margin2b">Bernoulli</a>, as described in 
<a style="color:#3c9591" href="https://arpm.co/lab/redirect.php?permalink=exm-unsup-clus-toy">this</a> example.

<h1 style="font-weight: bold; font-size:14px; line-height:1.5em;">Prepare environment</h1>

In [1]:
from numpy import abs, average, histogram, max, mean, var, sqrt
from numpy import linspace, argmin, exp, pi
from numpy.random import binomial, normal, seed
from sklearn.cluster import k_means
import pandas as pd

In [2]:
# allowing for the code to be reproducible, controlling binomial() and normal()
seed(42)

<h1 style="font-weight: bold; font-size:14px; line-height:1.5em;"> <span style="color:#f56502; font-weight:bold">[10 points]</span> Task 1</h1> 

Generate $\bar{\jmath}=10^{5}$ Monte Carlo simulations $\{x^{(j)}\}_{j=1}^{\bar{\jmath}}$ from the observable target variable $X$ 
according to the following setup for the parameters

\begin{equation}
\mu_{X|H}(0)\equiv0,\mu_{X|H}(1)\equiv2,\quad\sigma_{X|H}^{2}(0)\equiv
0.5,\sigma_{X|H}^{2}(1)\equiv0.2,\quad p_{H}\equiv0.4\text{.}
\end{equation}

To this purpose, simulate first the Bernoulli variable 
$\{h^{(j)}\}_{j=1}^{\bar{\jmath}}\overset{\text{MC}}{\Leftarrow}\mathit{Bernoulli}(p_{H})$ 
and the two mixtures 
$\{x^{(j)}|\{H=h\}\}_{j=1}^{\bar{\jmath}
}\overset{\text{MC}}{\Leftarrow}\operatorname{N}(\mu_{X|H}(h),\sigma_{X|H}
^{2}(h))$ for $h=0,1$; and then compute the target simulations as follows

\begin{equation}
x^{(j)}\equiv (1-h^{(j)})\times x^{(j)}|\{H=0\}+h^{(j)}\times x^{(j)}|\{H=1\}
\end{equation}

for any $j=1,\ldots ,\bar{\jmath}$. 

In [3]:
###################### inputs ######################
# number of Monte Carlo simulations
j_bar = 100000

# conditional expectations (2 means/center points of the same variable X, 1 per regime)
# marking the mean/center point of each bell/distribution
# Gap between the means = 0 - 2 => 2
mu_x_0 = 0
mu_x_1 = 2

# conditional variances (this indicates how wide each bell is)
# variance 0.5 is a standard deviation sqrt(0.5) = 0.707 (wide bell)
# variance 0.2 is a standard deviation sqrt(0.2) = 0.447 (narrow bell)
sigma2_x_0 = 0.5
sigma2_x_1 = 0.2
# parameter of Bernoulli distribution, probability coin lands on 1 -> (P{H=1})
p_h = 0.4
####################################################
    
# Bernoulli scenarios of hidden variable (array of 0, or 1 values(coin flips 100,000 in total))
h = binomial(1, p_h, j_bar)

# mixture scenarios - normal(mean, standard_deviation, how_many)
# standard_deviation is calculated as the sqrt(variance)
# normal takes (mean, standard_deviation, size_of_observations)
x_h0 = normal(mu_x_0, sqrt(sigma2_x_0), j_bar)
x_h1 = normal(mu_x_1, sqrt(sigma2_x_1), j_bar)

# target scenarios of observable variable
x =  (1 - h) * x_h0 + h * x_h1

# Check printouts
print("P{H=1}:", h.mean().round(4), "  target 0.4")
print("V{X|H=0}:", x[h == 0].var().round(4), "  target 0.5")
print("V{X|H=1}:", x[h == 1].var().round(4), "  target 0.2")

P{H=1}: 0.4   target 0.4
V{X|H=0}: 0.5009   target 0.5
V{X|H=1}: 0.2015   target 0.2


<h1 style="font-weight: bold; font-size:14px; line-height:1.5em;"> <span style="color:#f56502; font-weight:bold">[6 points]</span> Task 2</h1> 

Compute the 
<a style="color:#3c9591" href="https://arpm.co/lab/redirect.php?permalink=k-means-prob" class="@formulas #k-means-prob">optimal clustering</a> $\mathcal{C}^{2\mathit{-means}}$ which follows from the target simulations $\{x^{(j)}\}_{j=1}^{\bar{\jmath}}$.

<b>Hint</b>: Use function k_means
<a style="color:#3c9591" href="https://scikit-learn.org/stable/modules/generated/sklearn.cluster.k_means.html">[W]</a>. 

In [4]:
# run k-means algorithm
# reshape(-1,1) turns x from (100000,) into (100000, 1)
# runs 2-means 10 times from different random starts and keeps the best
centroid, label, _ = k_means(x.reshape(-1, 1), 2, n_init=10)

# ensure that samples are properly ordered and labeled
# if not then swaps everything
if centroid[1][0] < centroid[0][0]:
    label = 1 - label
    # swap 2 centroids
    centroid[0][0], centroid[1][0] = centroid[1][0], centroid[0][0]

# check
print("centroids:", centroid.ravel().round(4))
print()
print("share labelled 1:", label.mean().round(4))

centroids: [-0.141   1.8929]

share labelled 1: 0.4628


<h1 style="font-weight: bold; font-size:14px; line-height:1.5em;"> <span style="color:#f56502; font-weight:bold">[6 points]</span> Task 3</h1> 

Consider the 
<a style="color:#3c9591" href="https://arpm.co/lab/redirect.php?permalink=clust-identifier" class="@formulas #clust-identifier">optimal encoder</a>
\begin{equation}
\zeta _{\mathcal{C}^{2\mathit{-means}}}(x)\equiv \left\{ 
\begin{array}{ll}
1 & \text{for }x>x_{\mathcal{C}}^{\ast } \\ 
0 & \text{otherwise}
\end{array}\right.
\end{equation}

where $x_{\mathcal{C}}^{\ast }$ is the boundary point corresponding to the
<a style="color:#3c9591" href="https://arpm.co/lab/redirect.php?permalink=k-means-prob" class="@formulas #k-means-prob">optimal clustering</a>
$\mathcal{C}^{2\mathit{-means}}$ obtained in <b>Task 2</b>. Compute the simulations $\{z^{(j)}\equiv \zeta _{\mathcal{C}^{2}}(x^{(j)})\}_{j=1}^{\bar{\jmath}}$ for the 
<a style="color:#3c9591" href="https://arpm.co/lab/redirect.php?permalink=code-clust" class="@formulas #code-clust">code</a>
$Z\equiv\zeta _{\mathcal{C}^{2\mathit{-means}}}(X)$ stemming from the target simulations $\{x^{(j)}\}_{j=1}^{\bar{\jmath}}$ obtained in <b>Task 1</b>. Then use these simulations to compute the respective simulations $\{x^{(l)}|\{Z=0\}\}_{l=1}^{\bar{\jmath}-\bar{l}}$ and $\{x^{(l)}|\{Z=1\}\}_{l=1}^{\bar{l}}$ for the conditional observable target variable $X|z$.

In [5]:
# boundary/cutoff point 'x_star_c' of the optimal 2-means clustering
# in 1-D each point joins the nearer centroid, so the cutoff is the midpoint
x_star_c = average([centroid[0][0], centroid[1][0]])

# optimal encoder if v > x_star_c
zeta = lambda v: v > x_star_c

# simulations of the code Z = zeta(X): one 0/1 per simulated x
z = zeta(x)

# simulations of the conditional target variables X|{Z=0} and X|{Z=1}
# the points that fall at each side of the cutoff
x_z_0 = x[z==0]
x_z_1 = x[z==1]

# check
print("boundary x_star_c:", round(x_star_c, 4))
print()
print("points with z=0:", x_z_0.size, ", z =1:", x_z_1.size, ", total:", x_z_0.size + x_z_1.size)
print()
print("z agrees with k-means labels:", (z == label).mean())

boundary x_star_c: 0.876

points with z=0: 53718 , z =1: 46282 , total: 100000

z agrees with k-means labels: 1.0


<h1 style="font-weight: bold; font-size:14px; line-height:1.5em;"> <span style="color:#f56502; font-weight:bold">[10 points]</span> Task 4</h1>

Consider the 
<a style="color:#3c9591" href="https://arpm.co/lab/redirect.php?permalink=best-decoder" class="@formulas #best-decoder">optimal encoder</a>
\begin{equation}
\chi ^{\ast }(z)\equiv \left\{ 
\begin{array}{ll}
\mathbb{E}\{X|Z=1\} & \text{for }z=1 \\ 
\mathbb{E}\{X|Z=0\} & \text{for }z=0
\end{array}
\right.
\end{equation}

Use the simulations computed in <b>Task 3</b> to estimate $\chi^{\ast}(0), \chi^{\ast}(1)$, the probability $p\equiv \mathbb{P}\{Z=1\}$ and the respective
<a style="color:#3c9591" href="https://arpm.co/lab/redirect.php?permalink=k-means-constr-var" class="@formulas #k-means-constr-var">error</a>

\begin{equation}
e(x_{\mathcal{C}}^{\ast}, \chi^{\ast}(0), \chi^{\ast}(1))\equiv (1-p)\mathbb{E}\{(X-\chi ^{\ast }(0))^{2}|Z=0\}+p\mathbb{E}\{(X-\chi ^{\ast}(1))^{2}|Z=1\}.
\end{equation}

In [6]:
# mean of each cluster's points
e_x_z0 = mean(x_z_0)
e_x_z1 = mean(x_z_1)

# optimal decoder chi*(z)
# chi_star(0) = E{X|Z=0}
chi_star_0 = e_x_z0
# chi_star(1) = E{X|Z=1}
chi_star_1 = e_x_z1

# the decoded value for every simulation: x_hat = chi_star_z, from Task 1
chi_star_z = (1 - z) * chi_star_0 + z * chi_star_1

# probability, p = P{Z=1}, the share of points coded 1
p = mean(z)

# the reconstruction error e(x_star_c, chi_star_0, chi_star_1)
# (1-p) * E{(X - chi_star_0)^2 | Z=0} + p * E{(X - chi_star_1)^2 | Z=1}
e_xc_chi0_chi1 = (1 - p) * mean((x_z_0 - chi_star_0)**2) + p * mean((x_z_1 - chi_star_1)**2)

##################### result #####################
print('e_xc_chi0_chi1 =', e_xc_chi0_chi1.round(4))
print()
# check my addition
print("chi_star_0 = ", round(chi_star_0, 4), " chi_star_1 = ", round(chi_star_1, 4))
print()
print("p = ", round(p, 4))
print()
print("check: mean((x - chi_star_z)**2) = ", round(mean((x - chi_star_z)**2),4))

e_xc_chi0_chi1 = 0.3063

chi_star_0 =  -0.1411  chi_star_1 =  1.8927

p =  0.4628

check: mean((x - chi_star_z)**2) =  0.3063


<h1 style="font-weight: bold; font-size:14px; line-height:1.5em;"> <span style="color:#f56502; font-weight:bold">[6 points]</span> Task 5</h1>

Use the results obtained in <b>Task 4</b> to estimate the 
<a style="color:#3c9591" href="https://arpm.co/lab/redirect.php?permalink=law-tota-var" class="@formulas #law-tota-var">between-cluster variance</a>
$\mathbb{V}\{\mathbb{E\{}X|Z\}\}$ and the
<a style="color:#3c9591" href="https://arpm.co/lab/redirect.php?permalink=law-tota-var" class="@formulas #law-tota-var">within-cluster variance</a>
$\mathbb{E}\{\mathbb{V\{}X|Z\}\}$.

In [7]:
# total variance V{X}: spread of all simulated points
v_x = var(x)

# conditional variances
cv_x_z0 = var(x_z_0)
cv_x_z1 = var(x_z_1)

# within-cluster variance E{V{X|Z}}: cluster variances weighted by cluster size
e_var_x_z = (1 - p) * cv_x_z0 + p * cv_x_z1

# between-cluster variance V{E{X|Z}}: what's left of the total
var_e_x_z = v_x - e_var_x_z

################ results ################
print('var_e_x_z =', var_e_x_z.round(4))
print()
print('e_var_x_z =', e_var_x_z.round(4))
print()

# check: between-cluster variance computed directly
# as the variance of the decoded values chi_star_z from task 4 above
print("check: var(chi_star_z) =", var(chi_star_z).round(4))

var_e_x_z = 1.0284

e_var_x_z = 0.3063

check: var(chi_star_z) = 1.0284


<h1 style="font-weight: bold; font-size:14px; line-height:1.5em;"> <span style="color:#f56502; font-weight:bold">[12 points]</span> Task 6</h1>

Verify that the conditional variable $X|\{Z=z\}$ approximates better
the true mixture components $X|\{H=h\}$ when the mixture components $X|\{H=h\}$ are more separated.

Quantify this intuition by computing:

- The proportion of between-cluster variance over the total variance
\begin{equation}
\sigma_{\mathit{between}}^{2}\equiv \frac{\mathbb{V}\{\mathbb{E\{}X|Z\}\}}{\mathbb{V}\{X\}}
\end{equation}

In [8]:
# proportion of between-cluster variance over total variance
sigma2_between = var_e_x_z / v_x

- The proportion of within-cluster variance over the total variance 
\begin{equation}
\sigma_{\mathit{within}}^{2}\equiv \frac{\mathbb{E}\{\mathbb{V\{}X|Z\}\}}{\mathbb{V}\{X\}}
\end{equation}

In [9]:
# proportion of within-cluster variance over total variance
sigma2_within = e_var_x_z / v_x

- The relative sup distances
[<a style="color:#3c9591" href="https://en.wikipedia.org/wiki/Uniform_norm">W</a>] between the pdf's, i.e.
\begin{align}
\delta_{0}  &  \equiv\frac{\max_{x}|f(x|Z=0)-f(x|H=0)|}{\max_{x}|f(x|H=0)|} \\
\delta_{1}  &  \equiv\frac{\max_{x}|f(x|Z=1)-f(x|H=1)|}{\max_{x}|f(x|H=1)|}
\end{align}
as approximated by the respective histograms.

In [10]:
# one common set of bin edges, so every histogram is compared bar-for-bar
_, bins = histogram(x, bins=100)

# conditional pdfs
# f(x|Z=0): k-means cluster = 0
f_x_z_0, _ = histogram(x_z_0, bins=bins, density=True)
# f(x|Z=1): k-means cluster = 1
f_x_z_1, _ = histogram(x_z_1, bins=bins, density=True)
# f(x|H=0): true bell 0
f_x_h_0, _ = histogram(x[h == 0], bins=bins, density=True)
# f(x|H=1): true bell 1
f_x_h_1, _ = histogram(x[h == 1], bins=bins, density=True)

# relative sup distances between pdf's
# biggest bar by bar gap, relative to true bell's peak
delta_0 = max(abs(f_x_z_0 - f_x_h_0)) / max(f_x_h_0)
delta_1 = max(abs(f_x_z_1 - f_x_h_1)) / max(f_x_h_1)

##################### results #####################
print('sigma2_between =', sigma2_between.round(4))
print()
print('sigma2_within =', sigma2_within.round(4))
print()
print('delta_0 =', delta_0.round(4))
print()
print('delta_1 =', delta_1.round(4))

sigma2_between = 0.7705

sigma2_within = 0.2295

delta_0 = 0.4086

delta_1 = 0.3217


In [11]:
# Task 6 verification (my addition): repeating Tasks 1-6 for wider separations ----
def running_tasks_1_6_verification(mu_1):
    """Simulate the mixture with bell 1 centred at mu_1, run 2-means, return Task 6's measures."""
    # Task 1: simulate (same parameters as the inputs block, except mu_1)
    h_ = binomial(1, p_h, j_bar)
    x_ = (1 - h_)*normal(mu_x_0, sqrt(sigma2_x_0), j_bar) + h_*normal(mu_1, sqrt(sigma2_x_1), j_bar)
    # Tasks 2-3: k-means, boundary (midpoint of centroids: order doesn't matter), code
    c, _, _ = k_means(x_.reshape(-1, 1), 2, n_init=10)
    x_star = c.mean()
    z_ = x_ > x_star
    # Tasks 4-5: within-cluster variance
    p_ = z_.mean()
    within = (1 - p_)*var(x_[~z_]) + p_*var(x_[z_])
    # Task 6: sup distances on common bins
    _, bins = histogram(x_, bins=100)
    f = lambda s: histogram(s, bins=bins, density=True)[0]
    d0 = max(abs(f(x_[~z_]) - f(x_[h_ == 0]))) / max(f(x_[h_ == 0]))
    d1 = max(abs(f(x_[z_]) - f(x_[h_ == 1]))) / max(f(x_[h_ == 1]))
    return {"gap mu_1 - mu_0": mu_1 - mu_x_0, "cutoff x*_C": x_star,
            "sigma2_between": 1 - within/var(x_), "sigma2_within": within/var(x_),
            "delta_0": d0, "delta_1": d1, "accuracy vs H": (z_ == h_).mean()}

results = pd.DataFrame([running_tasks_1_6_verification(m) for m in [1, 2, 3, 4, 6]])
print(results.round(4).to_string(index=False))

 gap mu_1 - mu_0  cutoff x*_C  sigma2_between  sigma2_within  delta_0  delta_1  accuracy vs H
               1       0.2897          0.6824         0.3176   0.8405   0.4707         0.7735
               2       0.8737          0.7721         0.2279   0.4503   0.3613         0.9325
               3       1.4651          0.8569         0.1431   0.1016   0.0954         0.9886
               4       1.9903          0.9102         0.0898   0.0151   0.0147         0.9985
               6       3.0009          0.9575         0.0425   0.0000   0.0000         1.0000


In [12]:
# Extra verification, where the weighted bells cross
grid = linspace(0, 2, 20001)

# normal pdf
dens = lambda g, m, s2: exp(-(g - m)**2 / (2*s2)) / sqrt(2*pi*s2)

gap_w = abs((1 - p_h)*dens(grid, mu_x_0, sigma2_x_0) - p_h*dens(grid, mu_x_1, sigma2_x_1))

print("weighted densities cross at x ≈", grid[argmin(gap_w)].round(3),
      "  vs k-means cutoff", round(x_star_c, 3))

weighted densities cross at x ≈ 1.217   vs k-means cutoff 0.876


### My Conclusion.

As the separation between the mixture components grows (gap 1 → 6 in the table above), the 2-means
code explains more of the total variance (sigma2_between 0.68 → 0.96), and the clustered distributions
converge to the true components (delta_0 0.85 → 0, delta_1 0.46 → 0; accuracy 77% → 100%).

The largest discrepancy always sits at the k-means boundary:
each cluster is a bell truncated by a hard cut, so whatever tail of one bell crosses the cutoff is
misassigned.
Wider separation leaves less tail at the cutoff.

With this homework’s parameters (gap 2), the k-means cutoff (≈0.87) sits left of the point where the
weighted densities cross (≈1.22),
because k-means uses distance only and ignores the components’ different widths and weights.
A mixture model fitted by EM would account for both.